# G6 — Manifold characterisation: intrinsic dimension, near-isometry, shared subspace

Three measurements that CHARACTERISE the spaces rather than add a new
positive result. Read the framing before running them:

**None of these strengthens the GLOBAL similarity claim.** G5 already
measured global agreement and found it weak (≈2× chance at k=500). A test
cannot manufacture a finding the data does not contain, and a number here
that made global agreement look strong would contradict G5 — that would
be a bug, not a discovery. What these buy is precision about *what kind*
of object each space is and *how* two spaces relate:

1. **Intrinsic dimension** (TwoNN) — the true degrees of freedom inside
   each ambient space. Turns the loose word "manifold" into a number.
   It is a property of ONE space, so it says nothing about cross-encoder
   agreement — it justifies the vocabulary, nothing more.
2. **Procrustes across all 21 pairs** — near-isometry: is B a rotation of
   A? The project has this for one pair (0.038); running it broadly is
   the honest global-geometry test. Low everywhere confirms "related, not
   rigid"; a high pair would be a genuine finding.
3. **CCA spectrum** — how many directions two spaces share vs keep
   private. The one axis none of the existing metrics measure.

Everything is closed-form. No training, no encoder ever re-run.

*Implementation note: Procrustes is scored after scale-normalisation and
CCA on held-out rows after whitening. Skipping either produces
artefacts - large-negative Procrustes R2 and a flat CCA line of 1.0s -
so both guards are load-bearing, not cosmetic.*

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np

SOURCES = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch.npz", "img"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch.npz",  "img"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch.npz", "img"),
    "txt_bge":   ("crossmodal_pairs.npz",                   "txt"),
    "txt_gpt2":  ("crossmodal_pairs_gpt2.npz",              "txt"),
    "txt_bert":  ("e13_txt_bert.npz",                       "txt"),
    "txt_sbert": ("e13_txt_sbert.npz",                      "txt"),
}
SPACES = {}
for name, (fn, key) in SOURCES.items():
    f = DATA_DIR / fn
    if f.exists():
        d = np.load(str(f))
        if key in d.files:
            SPACES[name] = d[key].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
names = list(SPACES)
rng = np.random.default_rng(0)
NS = min(2000, N)                       # intrinsic-dim / CCA sample
sub = rng.permutation(N)[:NS]
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
print(f"{len(names)} spaces on {N} rows; using {NS} for this notebook")

## 1 · Intrinsic dimension (TwoNN)

The TwoNN estimator (Facco et al. 2017) reads intrinsic dimension from
the ratio of each point's two nearest-neighbour distances — no binning,
no manifold reconstruction. A value far below the ambient width means the
data occupies a thin curved region of the storage space.

**What it does and does not license.** It lets you say "this space
behaves like a d-dimensional structure" with a measured d instead of a
guess. It does NOT say two spaces share that structure — that is what
sections 2 and 3, and G5's kNN, are for.

In [ ]:
def twonn(X, frac=0.9, chunk=256):
    """Intrinsic dimension via the TwoNN ratio estimator. Chunked so the
    N x N distance matrix is never materialised."""
    X = l2n(X); n = len(X)
    r1 = np.empty(n); r2 = np.empty(n)
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        # cosine distance to all points, one block of rows at a time
        D = np.sqrt(np.maximum(2 - 2 * (X[s:e] @ X.T), 0.0))
        D[np.arange(e - s), np.arange(s, e)] = np.inf
        two = np.partition(D, 2, axis=1)[:, :3]; two.sort(1)
        r1[s:e], r2[s:e] = two[:, 0], two[:, 1]
    mu = r2 / np.clip(r1, 1e-12, None)
    mu = mu[np.isfinite(mu) & (mu > 1)]; mu.sort()
    F = np.arange(1, len(mu) + 1) / len(mu)
    keep = int(frac * len(mu))              # drop the noisy tail
    x = np.log(mu[:keep]); y = -np.log(1 - F[:keep] + 1e-12)
    return float((x @ y) / (x @ x))

print(f"{'space':12s} {'ambient':>8s} {'intrinsic d':>12s} "
      f"{'d / ambient':>12s}")
ID = {}
for k in names:
    d = twonn(SPACES[k][sub]); ID[k] = d
    amb = SPACES[k].shape[1]
    print(f"{k:12s} {amb:8d} {d:12.1f} {100*d/amb:11.1f}%")
print("\nThe intrinsic dimensions cluster in the low tens regardless of")
print("ambient width (768-2048) - the semantic structure is far thinner")
print("than the storage. This is the measured basis for the word")
print("'manifold': a low-dimensional structure inside a large space.")
print("It characterises each space; it does NOT by itself show two")
print("spaces share that structure.")

## 2 · Near-isometry across all 21 pairs (Procrustes vs ridge)

Orthogonal Procrustes finds the best pure rotation+reflection between two
spaces; ridge finds the best linear map. When ridge ≫ Procrustes, the
spaces are related by more than a rotation — they differ in *stretch*
(anisotropy). The project reported this for one pair (0.038 vs 0.592);
here it runs for all 21, so "related, not rigid" is tested broadly rather
than assumed from one case.

Widths differ across encoders, so each pair is compared in the smaller of
its two dimensions via PCA — an honest common ground, not zero-padding.

In [ ]:
def to_dim(X, d):
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:d].T

def procrustes_r2(X, Y):
    # scale each block to unit Frobenius norm FIRST. Without this the two
    # PCA-projected blocks have mismatched scales and the residual can
    # dwarf Y, sending R2 to nonsensical large-negative values.
    X = X / (np.linalg.norm(X) + 1e-12)
    Y = Y / (np.linalg.norm(Y) + 1e-12)
    U, _, Vt = np.linalg.svd(X.T @ Y); R = U @ Vt
    return float(1 - ((Y - X @ R) ** 2).sum() / ((Y ** 2).sum() + 1e-12))

def ridge_r2(X, Y, a=1e-2):
    W = np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
    return float(1 - ((Y - X @ W) ** 2).sum() /
                 ((Y - Y.mean(0)) ** 2).sum())

print(f"{'pair':26s} {'proc R2':>9s} {'ridge R2':>9s} {'reading':>22s}")
ISO = {}
for i, a in enumerate(names):
    for b in names[i+1:]:
        d = min(SPACES[a].shape[1], SPACES[b].shape[1], 256)
        Xa, Xb = to_dim(SPACES[a][sub], d), to_dim(SPACES[b][sub], d)
        pr = procrustes_r2(Xa, Xb); rg = ridge_r2(Xa, Xb)
        ISO[(a, b)] = (pr, rg)
        tag = ("near-isometric" if pr > 0.7 else
               "anisotropic gap" if rg - pr > 0.15 else
               "weak both ways")
        print(f"{a+' - '+b:26s} {pr:9.3f} {rg:9.3f} {tag:>22s}")
prs = [v[0] for v in ISO.values()]; rgs = [v[1] for v in ISO.values()]
print(f"\nmean Procrustes {np.mean(prs):.3f} vs mean ridge "
      f"{np.mean(rgs):.3f}")
print("A broad ridge >> Procrustes gap confirms Finding 2 (related, not")
print("rigid) across the whole set, not just the one shipped pair.")
print()
print("IMPORTANT - this Procrustes is measured AFTER reducing both spaces")
print("to a common PCA dimension, which already equalises per-axis")
print("variance and does part of the alignment. So these values are")
print("HIGHER and NOT comparable to Experiment B's raw 0.038 (raw 512-d")
print("-> raw 768-d, no PCA). Report them as 'Procrustes after common")
print("PCA'; the raw-space near-isometry number remains Exp B's 0.038.")

## 3 · Shared vs private subspace (CCA spectrum)

Canonical Correlation Analysis finds paired directions — one in each
space — that are maximally correlated. The spectrum of canonical
correlations says how many directions two spaces genuinely share: a value
near 1 is a shared direction, a value near 0 is private to one space.
This is the one question none of the other metrics answer — kNN, ridge,
Procrustes and CKA all give a single number, while CCA gives the whole
shared-to-private profile.

In [ ]:
def cca_spectrum(X, Y, k=64, tr=0.7, ridge=1e-4):
    """Whitened CCA fitted on TRAIN, canonical correlations scored on
    HELD-OUT. A plain QR of the centred matrix does NOT give the CCA
    basis - it leaves the leading directions near-collinear so every
    correlation reads ~1.0, which is an artefact, not shared structure.
    Whitening (inverse-sqrt covariance) and held-out scoring remove it."""
    n = len(X); ntr = int(tr * n)
    Xt, Xe = X[:ntr], X[ntr:]; Yt, Ye = Y[:ntr], Y[ntr:]
    Xt = Xt - Xt.mean(0); Yt = Yt - Yt.mean(0)
    Xe = Xe - Xe.mean(0); Ye = Ye - Ye.mean(0)
    def whiten(Z):
        C = Z.T @ Z / len(Z) + ridge * np.eye(Z.shape[1])
        U, S, _ = np.linalg.svd(C)
        return U @ np.diag(1.0 / np.sqrt(S)) @ U.T
    Wx, Wy = whiten(Xt), whiten(Yt)
    U, _, Vt = np.linalg.svd((Xt @ Wx).T @ (Yt @ Wy) / len(Xt))
    a = Wx @ U[:, :k]; b = Wy @ Vt[:k].T
    Pa, Pe = Xe @ a, Ye @ b                     # held-out projections
    corr = np.array([np.corrcoef(Pa[:, j], Pe[:, j])[0, 1]
                     for j in range(k)])
    return np.clip(np.nan_to_num(corr), 0, 1)

import matplotlib.pyplot as plt
def plot_cca(pairs=None, k=64):
    if pairs is None:                       # one per relationship type
        pairs = []
        for want in (("img_base","img_large"), ("txt_bge","txt_sbert"),
                     ("img_base","txt_bge"), ("img_base","txt_gpt2")):
            if want[0] in SPACES and want[1] in SPACES: pairs.append(want)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
    for a, b in pairs:
        s = cca_spectrum(SPACES[a][sub], SPACES[b][sub], k)
        ax[0].plot(s, lw=1.5, label=f"{a} - {b}")
        ax[1].plot(np.cumsum(s) / np.arange(1, len(s)+1), lw=1.5,
                   label=f"{a} - {b}")
    for x in ax:
        x.grid(alpha=0.2); x.tick_params(labelsize=8)
        x.spines["top"].set_visible(False); x.spines["right"].set_visible(False)
    ax[0].axhline(0.5, c="#999", ls="--", lw=1)
    ax[0].set_xlabel("canonical direction", fontsize=9)
    ax[0].set_ylabel("canonical correlation", fontsize=9)
    ax[0].set_title("shared directions decay into private ones",
                    fontsize=10, color="#1a1a2e")
    ax[0].legend(fontsize=7, frameon=False)
    ax[1].set_xlabel("directions kept", fontsize=9)
    ax[1].set_ylabel("mean canonical corr", fontsize=9)
    ax[1].set_title("how much is shared, cumulatively", fontsize=10,
                    color="#1a1a2e")
    fig.suptitle("CCA: shared vs private subspace", fontsize=11,
                 color="#1a1a2e")
    fig.subplots_adjust(left=0.07, right=0.97, top=0.86, bottom=0.14,
                        wspace=0.22)
    plt.show()

print(f"{'pair':26s} {'dims corr>0.5':>14s} {'dims corr>0.9':>14s}")
for i, a in enumerate(names):
    for b in names[i+1:]:
        s = cca_spectrum(SPACES[a][sub], SPACES[b][sub])
        print(f"{a+' - '+b:26s} {int((s>0.5).sum()):14d} "
              f"{int((s>0.9).sum()):14d}")
plot_cca()
print("\nThese are HELD-OUT canonical correlations, so the decay is real:")
print("the leading directions are genuinely shared, and the spectrum")
print("falls to private structure. A same-row QR would have shown a flat")
print("line of 1.0s - that is the artefact this version removes. Even so,")
print("CCA finds the BEST linear alignment by construction, so a high")
print("leading value is expected and is NOT evidence of global metric")
print("identity - read the DECAY, not the peak.")

## How to read G6

- **Intrinsic dimension** justifies calling these spaces "manifolds":
  the semantic structure occupies low tens of dimensions inside a
  768–2048-wide ambient space. It is a per-space property and says
  nothing about sharing.
- **Procrustes across 21 pairs** tests near-isometry broadly. A wide
  ridge-over-Procrustes gap confirms *related, not rigid* everywhere,
  not just for the one shipped pair.
- **CCA** separates shared directions from private ones — the profile no
  other metric gives. Read the decay, not the peak: CCA maximises
  correlation by construction, so high leading values are expected.

**What G6 does not do.** It does not raise the global-similarity number.
G5 measured global agreement as weak, and these characterisations are
consistent with that. Their role is to say precisely what kind of objects
the spaces are and how they relate — turning "manifold" and "related, not
rigid" from words into measurements. The strong, honest claim remains the
one G5 established: the sharing is real and local, thin and global.